## 空间数据可视化
python版本：3.7

核心第三方包：[geopandas](http://geopandas.org/) & [contextily](https://github.com/darribas/contextily)

数据：

- [shan3xi.json](data/shan3xi.json)：陕西省行政区划geojson格式文件
- [pois.txt](data/pois.txt): 陕西省内旅游景点数据，有噪音，坐标系4326 (经度字段：lng，维度字段：lat)


%matplotlib inline 用于使用matplotlib在该页面内绘制图表

In [1]:
%matplotlib inline

### Geopandas读取GeoJson并绘制

In [1]:
import geopandas as gpd

In [3]:
fp = "./data/shan3xi.json"

In [4]:
shaanxi = gpd.read_file(fp)

In [ ]:
type(shaanxi)

In [ ]:
shaanxi.head()

In [ ]:
ax = shaanxi.plot(figsize=(10, 10), alpha=0.5, edgecolor='k')

### Pandas读取CSV文件，添加几何信息，转换为GeoPandas GeoDataFrame 并去除噪音

In [8]:
import pandas as pd

In [9]:
from shapely.geometry import Point

In [10]:
pois = pd.read_csv("./data/pois.txt")

In [ ]:
pois.head()

In [12]:
pois["geometry"]=pois.apply(lambda z:Point(z.lng,z.lat),axis=1)

In [13]:
pois = gpd.GeoDataFrame(
    pois, crs  ={'init': 'epsg:4326'}
)

In [ ]:
pois.head()

In [ ]:
xmin, xmax, ymin, ymax = ax.axis()
xmin, xmax, ymin, ymax

In [ ]:
pois.plot(figsize=(10, 10), alpha=0.5, edgecolor='k')

In [ ]:
pois=pois.loc[(pois.lng>xmin)&(pois.lng<xmax)&(pois.lat>ymin)&(pois.lat<ymax)]

In [ ]:
pois.head()

In [ ]:
pois.plot(figsize=(10, 10), alpha=0.5, edgecolor='k')

### 绘制空间数据并添加底图（epsg=3857）
注：底图空间参考为web墨卡托，获取底图可能会有点慢

In [20]:
import contextily as ctx

In [21]:
def add_basemap(ax, zoom, url):
    xmin, xmax, ymin, ymax = ax.axis()
    basemap, extent = ctx.bounds2img(
        xmin, ymin, xmax, ymax, zoom=zoom, url=url)
    ax.imshow(basemap, extent=extent, interpolation='bilinear')
    ax.axis((xmin, xmax, ymin, ymax))

In [ ]:
ax = shaanxi.to_crs(epsg=3857).plot(figsize=(10, 10), alpha=0.5, edgecolor='k')
add_basemap(ax, zoom=10,url="http://webrd02.is.autonavi.com/appmaptile?x={x}&y={y}&z={z}&lang=zh_cn&size=1&scale=1&style=8")

In [ ]:
ax = shaanxi.to_crs(epsg=3857).plot(figsize=(10, 10), alpha=0.5, edgecolor='k')
pois.to_crs(epsg=3857).plot(ax=ax, alpha=0.5,edgecolor='green')
add_basemap(ax, zoom=10,url="http://webrd01.is.autonavi.com/appmaptile?x={x}&y={y}&z={z}&lang=zh_cn&size=1&scale=1&style=8")